In [1]:
pip install pandas scikit-learn skrebate

  Installing build dependencies .done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for skrebate: filename=skrebate-0.62-py3-none-any.whl size=29341 sha256=b508b665803a327e873d3ffbfa6376f1cb129799d3c074551bffd546a1dc8452
  Stored in directory: /Users/joshuakirubancruzs/Library/Caches/pip/wheels/f0/04/f4/1a6945c69819f1641bca4871bbec1345f872cdf8e19593e9bf
Successfully built skrebate
Note: you may need to restart the kernel to use updated packages.


In [2]:
#### 8 Selected Features  ####

import pandas as pd
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from skrebate import ReliefF

# ---------------------------------------------------------
# 1. Load and Prepare the Dataset
# ---------------------------------------------------------
print("Loading Breast Cancer dataset...")
cancer_dataset = datasets.load_breast_cancer()

# Create a DataFrame for easier viewing and manipulation
df = pd.DataFrame(data=cancer_dataset.data, columns=cancer_dataset.feature_names)
df['TARGET'] = cancer_dataset.target

# Separate features (X) and target (y)
# It's good practice to use the raw arrays for skrebate
X = df.drop('TARGET', axis=1).values
y = df['TARGET'].values

feature_names = np.array(df.drop('TARGET', axis=1).columns)

print(f"Original dataset shape: {X.shape}")
print(f"Total available features: {len(feature_names)}\n")

# ---------------------------------------------------------
# 2. Configure and Run ReliefF
# ---------------------------------------------------------
# n_features_to_select: How many top features you want to keep
# n_neighbors: How many nearest hits/misses to use for scoring (default is usually 100)
NUMBER_OF_FEATURES_TO_SELECT = 8

print(f"Running ReliefF to select the top {NUMBER_OF_FEATURES_TO_SELECT} features...")
selector = ReliefF(n_features_to_select=NUMBER_OF_FEATURES_TO_SELECT, n_neighbors=100)

# Fit the selector to the data. This calculates the feature weights.
selector.fit(X, y)

# ---------------------------------------------------------
# 3. Analyze the Results (Which features were selected?)
# ---------------------------------------------------------

# The 'feature_importances_' attribute holds the ReliefF score for every original feature
feature_scores = selector.feature_importances_

# We can create a DataFrame to view the scores neatly
scores_df = pd.DataFrame({
    'Feature': feature_names,
    'ReliefF_Score': feature_scores
})

# Sort the DataFrame by score in descending order (highest score = best feature)
scores_df = scores_df.sort_values(by='ReliefF_Score', ascending=False).reset_index(drop=True)

print("\n--- All Feature Scores (Ranked) ---")
print(scores_df)

# The 'top_features_' attribute holds the INDICES of the features sorted by importance
# We grab the first N indices corresponding to the features we want to select
top_n_indices = selector.top_features_[:NUMBER_OF_FEATURES_TO_SELECT]
selected_feature_names = feature_names[top_n_indices]

print(f"\n--- The Top {NUMBER_OF_FEATURES_TO_SELECT} Selected Features ---")
for i, name in enumerate(selected_feature_names):
    print(f"{i+1}. {name} (Score: {scores_df.loc[i, 'ReliefF_Score']:.4f})")

# ---------------------------------------------------------
# 4. Transform the Data
# ---------------------------------------------------------
# Now, actually reduce the dataset to only contain those top features
X_new = selector.transform(X)

print(f"\nNew dataset shape after transformation: {X_new.shape}")

# Optional: Put the transformed data back into a DataFrame so you know what you are looking at
df_reduced = pd.DataFrame(data=X_new, columns=selected_feature_names)
df_reduced['TARGET'] = y

print("\n--- Preview of Reduced Dataset ---")
pd.set_option('display.max_columns', None)
print(df_reduced.head())

Loading Breast Cancer dataset...
Original dataset shape: (569, 30)
Total available features: 30

Running ReliefF to select the top 8 features...

--- All Feature Scores (Ranked) ---
                    Feature  ReliefF_Score
0      worst concave points       0.175989
1       mean concave points       0.141696
2              worst radius       0.136938
3           worst perimeter       0.133236
4            mean perimeter       0.117638
5               mean radius       0.114942
6            mean concavity       0.111510
7                worst area       0.100071
8                 mean area       0.097906
9           worst concavity       0.091500
10        worst compactness       0.062963
11         mean compactness       0.062490
12            worst texture       0.050589
13         worst smoothness       0.046980
14             radius error       0.038746
15             mean texture       0.037813
16               area error       0.033937
17          perimeter error       0.033197
1

In [12]:
features = df[['worst concave points','mean concave points', 'worst radius','worst perimeter','mean perimeter','mean radius','mean concavity','worst area']]
label = df['TARGET']

In [16]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 5           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # Optional: Print the average across all folds at the end of the model evaluation
    print(f"\n*** Overall Summary for {name} ***")
    print(f"  Mean Train Accuracy: {np.mean(train_accs):.4f} ± {np.std(train_accs):.4f}")
    print(f"  Mean Test Accuracy : {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")

    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9143
  Test Accuracy : 0.9298
  Train Precision : 0.8946 ± 0.0000
  Test Precision  : 0.9200 ± 0.0000
  Train Recall    : 0.9790 ± 0.0000
  Test Recall     : 0.9718 ± 0.0000
  Train F1-Score  : 0.9349 ± 0.0000
  Test F1-Score   : 0.9452 ± 0.0000
  Train Confusion Matrix:
[[136  33]
 [  6 280]]
  Test Confusion Matrix:
[[37  6]
 [ 2 69]]

--- Fold 2 ---
  Train Accuracy: 0.9231
  Test Accuracy : 0.8772
  Train Precision : 0.8991 ± 0.0045
  Test Precision  : 0.8859 ± 0.0341
  Train Recall    : 0.9808 ± 0.0017
  Test Recall     : 0.9718 ± 0.0000
  Train F1-Score  : 0.9381 ± 0.0032
  Test F1-Score   : 0.9266 ± 0.0187
  Train Confusion Matrix:
[[139  30]
 [  5 281]]
  Test Confusion Matrix:
[[31 12]
 [ 2 69]]

--- Fold 3 ---
  Train Accuracy: 0.9121
  Test Accuracy : 0.8860
  Train Precision : 0.8973 ± 0.0044
  Test Precision  : 0.8730 ± 0.0333
  Train Recall    : 0.9790 ± 0

In [17]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 8           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # Optional: Print the average across all folds at the end of the model evaluation
    print(f"\n*** Overall Summary for {name} ***")
    print(f"  Mean Train Accuracy: {np.mean(train_accs):.4f} ± {np.std(train_accs):.4f}")
    print(f"  Mean Test Accuracy : {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")

    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9195
  Test Accuracy : 0.9167
  Train Precision : 0.9072 ± 0.0000
  Test Precision  : 0.9149 ± 0.0000
  Train Recall    : 0.9712 ± 0.0000
  Test Recall     : 0.9556 ± 0.0000
  Train F1-Score  : 0.9381 ± 0.0000
  Test F1-Score   : 0.9348 ± 0.0000
  Train Confusion Matrix:
[[154  31]
 [  9 303]]
  Test Confusion Matrix:
[[23  4]
 [ 2 43]]

--- Fold 2 ---
  Train Accuracy: 0.9116
  Test Accuracy : 0.9859
  Train Precision : 0.9020 ± 0.0052
  Test Precision  : 0.9463 ± 0.0314
  Train Recall    : 0.9712 ± 0.0000
  Test Recall     : 0.9778 ± 0.0222
  Train F1-Score  : 0.9353 ± 0.0028
  Test F1-Score   : 0.9618 ± 0.0270
  Train Confusion Matrix:
[[150  35]
 [  9 304]]
  Test Confusion Matrix:
[[26  1]
 [ 0 44]]

--- Fold 3 ---
  Train Accuracy: 0.9257
  Test Accuracy : 0.8310
  Train Precision : 0.9041 ± 0.0052
  Test Precision  : 0.9001 ± 0.0702
  Train Recall    : 0.9744 ± 0

In [18]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 10           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # Optional: Print the average across all folds at the end of the model evaluation
    print(f"\n*** Overall Summary for {name} ***")
    print(f"  Mean Train Accuracy: {np.mean(train_accs):.4f} ± {np.std(train_accs):.4f}")
    print(f"  Mean Test Accuracy : {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")

    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9199
  Test Accuracy : 0.9123
  Train Precision : 0.9049 ± 0.0000
  Test Precision  : 0.9167 ± 0.0000
  Train Recall    : 0.9752 ± 0.0000
  Test Recall     : 0.9429 ± 0.0000
  Train F1-Score  : 0.9387 ± 0.0000
  Test F1-Score   : 0.9296 ± 0.0000
  Train Confusion Matrix:
[[157  33]
 [  8 314]]
  Test Confusion Matrix:
[[19  3]
 [ 2 33]]

--- Fold 2 ---
  Train Accuracy: 0.9141
  Test Accuracy : 0.9649
  Train Precision : 0.9033 ± 0.0016
  Test Precision  : 0.9313 ± 0.0146
  Train Recall    : 0.9720 ± 0.0031
  Test Recall     : 0.9714 ± 0.0286
  Train F1-Score  : 0.9364 ± 0.0023
  Test F1-Score   : 0.9509 ± 0.0213
  Train Confusion Matrix:
[[156  34]
 [ 10 312]]
  Test Confusion Matrix:
[[20  2]
 [ 0 35]]

--- Fold 3 ---
  Train Accuracy: 0.9199
  Test Accuracy : 0.9123
  Train Precision : 0.9038 ± 0.0014
  Test Precision  : 0.9200 ± 0.0199
  Train Recall    : 0.9731 ± 0

In [4]:
#### optimal features Selected Features  ####

import pandas as pd
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from skrebate import ReliefF

# ---------------------------------------------------------
# 1. Load and Prepare the Dataset
# ---------------------------------------------------------
print("Loading Breast Cancer dataset...")
cancer_dataset = datasets.load_breast_cancer()

# Create a DataFrame for easier viewing and manipulation
df = pd.DataFrame(data=cancer_dataset.data, columns=cancer_dataset.feature_names)
df['TARGET'] = cancer_dataset.target

# Separate features (X) and target (y)
# It's good practice to use the raw arrays for skrebate
X = df.drop('TARGET', axis=1).values
y = df['TARGET'].values

feature_names = np.array(df.drop('TARGET', axis=1).columns)

print(f"Original dataset shape: {X.shape}")
print(f"Total available features: {len(feature_names)}\n")

# ---------------------------------------------------------
# 2. Configure and Run ReliefF
# ---------------------------------------------------------
# n_features_to_select: How many top features you want to keep
# n_neighbors: How many nearest hits/misses to use for scoring (default is usually 100)
NUMBER_OF_FEATURES_TO_SELECT = 15

print(f"Running ReliefF to select the top {NUMBER_OF_FEATURES_TO_SELECT} features...")
selector = ReliefF(n_features_to_select=NUMBER_OF_FEATURES_TO_SELECT, n_neighbors=100)

# Fit the selector to the data. This calculates the feature weights.
selector.fit(X, y)

# ---------------------------------------------------------
# 3. Analyze the Results (Which features were selected?)
# ---------------------------------------------------------

# The 'feature_importances_' attribute holds the ReliefF score for every original feature
feature_scores = selector.feature_importances_

# We can create a DataFrame to view the scores neatly
scores_df = pd.DataFrame({
    'Feature': feature_names,
    'ReliefF_Score': feature_scores
})

# Sort the DataFrame by score in descending order (highest score = best feature)
scores_df = scores_df.sort_values(by='ReliefF_Score', ascending=False).reset_index(drop=True)

print("\n--- All Feature Scores (Ranked) ---")
print(scores_df)

# The 'top_features_' attribute holds the INDICES of the features sorted by importance
# We grab the first N indices corresponding to the features we want to select
top_n_indices = selector.top_features_[:NUMBER_OF_FEATURES_TO_SELECT]
selected_feature_names = feature_names[top_n_indices]

print(f"\n--- The Top {NUMBER_OF_FEATURES_TO_SELECT} Selected Features ---")
for i, name in enumerate(selected_feature_names):
    print(f"{i+1}. {name} (Score: {scores_df.loc[i, 'ReliefF_Score']:.4f})")

# ---------------------------------------------------------
# 4. Transform the Data
# ---------------------------------------------------------
# Now, actually reduce the dataset to only contain those top features
X_new = selector.transform(X)

print(f"\nNew dataset shape after transformation: {X_new.shape}")

# Optional: Put the transformed data back into a DataFrame so you know what you are looking at
df_reduced = pd.DataFrame(data=X_new, columns=selected_feature_names)
df_reduced['TARGET'] = y

print("\n--- Preview of Reduced Dataset ---")
pd.set_option('display.max_columns', None)
print(df_reduced.head())

Loading Breast Cancer dataset...
Original dataset shape: (569, 30)
Total available features: 30

Running ReliefF to select the top 15 features...

--- All Feature Scores (Ranked) ---
                    Feature  ReliefF_Score
0      worst concave points       0.175989
1       mean concave points       0.141696
2              worst radius       0.136938
3           worst perimeter       0.133236
4            mean perimeter       0.117638
5               mean radius       0.114942
6            mean concavity       0.111510
7                worst area       0.100071
8                 mean area       0.097906
9           worst concavity       0.091500
10        worst compactness       0.062963
11         mean compactness       0.062490
12            worst texture       0.050589
13         worst smoothness       0.046980
14             radius error       0.038746
15             mean texture       0.037813
16               area error       0.033937
17          perimeter error       0.033197


In [19]:
# Optimal selection: Top 10 features based on ReliefF scores > 0.09
features = df[[
    'worst concave points', 
    'mean concave points', 
    'worst radius', 
    'worst perimeter', 
    'mean perimeter', 
    'mean radius', 
    'mean concavity', 
    'worst area', 
    'mean area', 
    'worst concavity'
]]
label = df['TARGET']

In [20]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 5           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # ---------------------------------------------------------
    # NEW: Expanded Overall Summary
    # ---------------------------------------------------------
    print(f"\n*** Overall Summary for {name} ***")
    
    # Helper to print mean ± std for each metric
    metrics_summary = [
        ("Accuracy ", train_accs, test_accs),
        ("Precision", train_precs, test_precs),
        ("Recall   ", train_recs, test_recs),
        ("F1-Score ", train_f1s, test_f1s)
    ]
    
    for metric_name, train_list, test_list in metrics_summary:
        print(f"  Mean Train {metric_name}: {np.mean(train_list):.4f} ± {np.std(train_list):.4f}")
        print(f"  Mean Test  {metric_name}: {np.mean(test_list):.4f} ± {np.std(test_list):.4f}")
        print("-" * 40)
    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9077
  Test Accuracy : 0.9211
  Train Precision : 0.8836 ± 0.0000
  Test Precision  : 0.9079 ± 0.0000
  Train Recall    : 0.9825 ± 0.0000
  Test Recall     : 0.9718 ± 0.0000
  Train F1-Score  : 0.9305 ± 0.0000
  Test F1-Score   : 0.9388 ± 0.0000
  Train Confusion Matrix:
[[132  37]
 [  5 281]]
  Test Confusion Matrix:
[[36  7]
 [ 2 69]]

--- Fold 2 ---
  Train Accuracy: 0.9077
  Test Accuracy : 0.8772
  Train Precision : 0.8824 ± 0.0012
  Test Precision  : 0.8756 ± 0.0323
  Train Recall    : 0.9843 ± 0.0017
  Test Recall     : 0.9789 ± 0.0070
  Train F1-Score  : 0.9306 ± 0.0001
  Test F1-Score   : 0.9239 ± 0.0148
  Train Confusion Matrix:
[[131  38]
 [  4 282]]
  Test Confusion Matrix:
[[30 13]
 [ 1 70]]

--- Fold 3 ---
  Train Accuracy: 0.9121
  Test Accuracy : 0.8860
  Train Precision : 0.8863 ± 0.0055
  Test Precision  : 0.8661 ± 0.0296
  Train Recall    : 0.9813 ± 0

In [21]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 8           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # ---------------------------------------------------------
    # NEW: Expanded Overall Summary
    # ---------------------------------------------------------
    print(f"\n*** Overall Summary for {name} ***")
    
    # Helper to print mean ± std for each metric
    metrics_summary = [
        ("Accuracy ", train_accs, test_accs),
        ("Precision", train_precs, test_precs),
        ("Recall   ", train_recs, test_recs),
        ("F1-Score ", train_f1s, test_f1s)
    ]
    
    for metric_name, train_list, test_list in metrics_summary:
        print(f"  Mean Train {metric_name}: {np.mean(train_list):.4f} ± {np.std(train_list):.4f}")
        print(f"  Mean Test  {metric_name}: {np.mean(test_list):.4f} ± {np.std(test_list):.4f}")
        print("-" * 40)
    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9115
  Test Accuracy : 0.9028
  Train Precision : 0.8941 ± 0.0000
  Test Precision  : 0.8958 ± 0.0000
  Train Recall    : 0.9744 ± 0.0000
  Test Recall     : 0.9556 ± 0.0000
  Train F1-Score  : 0.9325 ± 0.0000
  Test F1-Score   : 0.9247 ± 0.0000
  Train Confusion Matrix:
[[149  36]
 [  8 304]]
  Test Confusion Matrix:
[[22  5]
 [ 2 43]]

--- Fold 2 ---
  Train Accuracy: 0.9016
  Test Accuracy : 0.9859
  Train Precision : 0.8867 ± 0.0074
  Test Precision  : 0.9368 ± 0.0410
  Train Recall    : 0.9760 ± 0.0016
  Test Recall     : 0.9778 ± 0.0222
  Train F1-Score  : 0.9292 ± 0.0033
  Test F1-Score   : 0.9567 ± 0.0320
  Train Confusion Matrix:
[[143  42]
 [  7 306]]
  Test Confusion Matrix:
[[26  1]
 [ 0 44]]

--- Fold 3 ---
  Train Accuracy: 0.9237
  Test Accuracy : 0.8169
  Train Precision : 0.8922 ± 0.0099
  Test Precision  : 0.8887 ± 0.0758
  Train Recall    : 0.9787 ± 0

In [22]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 10           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # ---------------------------------------------------------
    # NEW: Expanded Overall Summary
    # ---------------------------------------------------------
    print(f"\n*** Overall Summary for {name} ***")
    
    # Helper to print mean ± std for each metric
    metrics_summary = [
        ("Accuracy ", train_accs, test_accs),
        ("Precision", train_precs, test_precs),
        ("Recall   ", train_recs, test_recs),
        ("F1-Score ", train_f1s, test_f1s)
    ]
    
    for metric_name, train_list, test_list in metrics_summary:
        print(f"  Mean Train {metric_name}: {np.mean(train_list):.4f} ± {np.std(train_list):.4f}")
        print(f"  Mean Test  {metric_name}: {np.mean(test_list):.4f} ± {np.std(test_list):.4f}")
        print("-" * 40)
    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9141
  Test Accuracy : 0.8947
  Train Precision : 0.8927 ± 0.0000
  Test Precision  : 0.8919 ± 0.0000
  Train Recall    : 0.9814 ± 0.0000
  Test Recall     : 0.9429 ± 0.0000
  Train F1-Score  : 0.9349 ± 0.0000
  Test F1-Score   : 0.9167 ± 0.0000
  Train Confusion Matrix:
[[152  38]
 [  6 316]]
  Test Confusion Matrix:
[[18  4]
 [ 2 33]]

--- Fold 2 ---
  Train Accuracy: 0.9043
  Test Accuracy : 0.9649
  Train Precision : 0.8886 ± 0.0041
  Test Precision  : 0.9189 ± 0.0270
  Train Recall    : 0.9783 ± 0.0031
  Test Recall     : 0.9714 ± 0.0286
  Train F1-Score  : 0.9313 ± 0.0036
  Test F1-Score   : 0.9444 ± 0.0278
  Train Confusion Matrix:
[[149  41]
 [  8 314]]
  Test Confusion Matrix:
[[20  2]
 [ 0 35]]

--- Fold 3 ---
  Train Accuracy: 0.9121
  Test Accuracy : 0.9123
  Train Precision : 0.8897 ± 0.0037
  Test Precision  : 0.9118 ± 0.0243
  Train Recall    : 0.9782 ± 0